In [4]:
import pandas as pd
from glob import glob
import sys
import os
import sklearn.neighbors._base
sys.modules['sklearn.neighbors.base'] = sklearn.neighbors._base

from tqdm.notebook import tqdm

from missingpy import MissForest

ImportError: cannot import name '_check_weights' from 'sklearn.neighbors._base' (c:\Users\hp\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neighbors\_base.py)

In [12]:
for file in tqdm(glob("*_0515.csv")):
    dataset_until_15 = pd.read_csv(file, sep=";", decimal=",", parse_dates=["FECHA_HORA"], index_col="FECHA_HORA")
    dataset_after_15 = pd.read_csv(file.split("_")[0]+"_1523.csv", sep=";", decimal=",", parse_dates=["FECHA_HORA"], index_col="FECHA_HORA")
    dataset_after_15 = dataset_after_15[(dataset_after_15.index.year>2015) & (dataset_after_15.index.year<=2023)]

    dataset_until_15 = dataset_until_15.drop(dataset_until_15.columns[-1], axis=1)

    dataset = pd.concat((dataset_until_15, dataset_after_15))

    dataset = dataset.loc[:, ~dataset.loc[dataset.index.year>=2022].isna().all(axis=0).values]

    imputer = MissForest(criterion="squared_error")
    train_inputed = imputer.fit_transform(dataset.loc[dataset.index.year<2022])
    dataset_train_inputed = pd.DataFrame(train_inputed, columns=dataset.columns, index=dataset.loc[dataset.index.year<2022].index)

    valtest_inputed = imputer.transform(dataset.loc[dataset.index.year>=2022])
    dataset_valtest_inputed = pd.DataFrame(valtest_inputed, columns=dataset.columns,index=dataset.loc[dataset.index.year>=2022].index)

    dataset_inputed = pd.concat((dataset_train_inputed, dataset_valtest_inputed))

    dataset_inputed.columns = [col.split("-")[1].split()[0].lower() for col in dataset_inputed.columns]

    dataset_inputed["year"] = dataset_inputed.index.year

    dataset_inputed = dataset_inputed[["year", *dataset_inputed.columns[:-1]]]

    data_folder = f"../../processed/{file.split('_')[0]}0523"
    if not os.path.isdir(data_folder):
        os.mkdir(data_folder)

    dataset_inputed.to_csv(f"{data_folder}/data.csv")

  0%|          | 0/5 [00:00<?, ?it/s]

Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 0
Iteration: 1
Iteration: 2
Iteration: 3
Iteration: 4


In [19]:
for file in glob(f"../../processed/*0523/data.csv"):

    data = pd.read_csv(file)
    o3_series = data["o3"]

    data.drop(["o3"], axis=1, inplace=True)


    data["target_o3"] = o3_series

    data.to_csv(file, index=None)


In [18]:
data

,year,dd,no2,pm10,tmp,vv,target_o3
0,2005,163.5000,71.0000,212.6667,15.0000,4.3333,10.5000
1,2005,163.5000,71.0000,212.6667,15.0000,4.3333,10.5000
2,2005,33.8333,45.1667,229.8333,15.5000,10.3333,7.6667
3,2005,54.6667,49.6667,100.8333,15.6667,7.6667,9.3333
4,2005,44.3333,52.3333,101.3333,15.0000,5.8333,9.6667
...,...,...,...,...,...,...,...
166531,2023,161.8333,23.6667,22.5000,14.6667,0.3333,36.3333
166532,2023,39.8333,36.1667,25.5000,12.6667,1.6667,14.6667
166533,2023,29.3333,28.3333,31.0000,11.1667,2.0000,18.1667
166534,2023,25.1667,29.1667,32.8333,11.0000,2.0000,18.3333
